# laya-guard: fine-tune Laya on a free GPU

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. Upload only `train/train.jsonl` and `train/val.jsonl` from the laya-guard checkout.
3. Download `laya-guard-ckpt.zip` at the end and unzip it next to the repo (`train/ckpt/`).

Training continues from the model laya-guard runs (cell 5 downloads it at the revision `guard.py` pins), so what it
already judges well is kept: trained again from base Laya, every attempt to add something new moved the shell verdicts
too. Set `BASE_REVISION` to `None` in cell 5 only to start over from base Laya.

The two files hold masked commands from personal projects. They live only in this session: nothing here saves them to
Drive or a dataset. The checkpoint is trained on them too, so keep it private.

In [ ]:
# 1. A GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. laya 0.4.3: the version laya-guard's shipped model was trained and measured with. Check yours with
#    `pip show laya` and pin the same here, so the checkpoint loads where you run laya-guard.
!pip install -q laya==0.4.3
!laya-train --help | head -5

In [ ]:
# 3. Upload train.jsonl and val.jsonl
from google.colab import files
up = files.upload()
for name in ("train.jsonl", "val.jsonl"):
    assert name in up, f"missing {name}"
    print(name, sum(1 for line in open(name) if line.strip()), "rows")

In [ ]:
# 4. Check the data without loading the model
!laya-train --data train.jsonl --eval val.jsonl --base multilingual --out ckpt --dry-run

In [ ]:
# 5. Train (prints the validation score before and after), starting from the model laya-guard runs.
# BASE_REVISION: keep it equal to GUARD_MODEL_REVISION in src/laya_guard/guard.py (a test checks). None starts from
# base Laya instead (then use about 4 epochs and the default learning rate). 2 epochs at encoder lr 1e-5 add to what the
# model knows without moving much of it. SEED: keep it fixed while you compare data changes; run a second seed on a
# candidate before you adopt it (1-2 cases either way can be chance).
# rm -rf ckpt: no checkpoint from an earlier run survives. pipefail: a failed laya-train isn't reported as tee's
# success. A `!` line doesn't stop the notebook, so cell 6 also refuses to package without the final model.
BASE_REPO = "Ayaannnnnnn/laya-guard-multilingual"
BASE_REVISION = "6c5d9ae69bb01f294c467fc460200ca3ada209ab"
SEED = 0
import shutil
shutil.rmtree("ckpt", ignore_errors=True)
if BASE_REVISION:
    from huggingface_hub import snapshot_download
    shutil.rmtree("base", ignore_errors=True)
    snapshot_download(BASE_REPO, revision=BASE_REVISION, local_dir="base",
                      allow_patterns=["model.safetensors", "rl_agent_config.json", "encoder/*", "tokenizer/*"])
    TRAIN_ARGS = "--base ./base --epochs 2 --encoder-lr 1e-5"
else:
    TRAIN_ARGS = "--base multilingual --epochs 4"
!bash -o pipefail -c "laya-train --data train.jsonl --eval val.jsonl --out ckpt --device cuda --seed {SEED} {TRAIN_ARGS} 2>&1 | tee train.log"
print("seed", SEED, "|", TRAIN_ARGS)

In [ ]:
# 6. Zip the checkpoint and the log, and download them
import shutil, os
assert os.path.exists("ckpt/model.safetensors"), "no final model: training did not finish (see train.log)"
shutil.copy("train.log", "ckpt/train.log")
shutil.make_archive("laya-guard-ckpt", "zip", "ckpt")
print(round(os.path.getsize("laya-guard-ckpt.zip") / 2**20), "MB")
files.download("laya-guard-ckpt.zip")

Optional, instead of downloading: push to a **private** Hugging Face repo.

```python
from huggingface_hub import notebook_login, HfApi
notebook_login()                      # paste a write token in the box, never in a cell
api = HfApi()
repo = api.create_repo("<your-name>/laya-guard-ckpt", private=True, exist_ok=True).repo_id
api.update_repo_settings(repo, private=True)   # create_repo leaves an existing public repo public
api.upload_folder(folder_path="ckpt", repo_id=repo)
```